# Position Sizing: Fixed & Risk-Based

quaver supports two sizing modes:

1. **Fixed sizing** -- a constant `quantity_per_trade` for every trade
2. **Dynamic sizing** -- a `sizing_fn(account_value, entry_price) -> quantity`
   callable that computes position size at each entry

The `size_by_risk()` utility function implements **risk-per-trade sizing**:
given a risk budget (fraction of account), entry price, and stop-loss level,
it computes the position size that limits loss to the risk budget.

## Formula

```
risk_per_unit = |entry_price - stop_loss|
quantity = (account_value * risk_pct) / risk_per_unit
```

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime, timedelta

from quaver.backtest.engine import BacktestEngine
from quaver.backtest.portfolio import ExitRules, Portfolio
from quaver.backtest.sizing import size_by_risk
from quaver.strategies.base import BaseStrategy, SignalOutput
from quaver.types import SignalDirection

%matplotlib inline
plt.rcParams["figure.figsize"] = (12, 4)
plt.rcParams["figure.dpi"] = 100

## 1. `size_by_risk()` Basics

Compute how many shares to buy to risk at most 2% of a $10,000 account
with an entry at $100 and a stop-loss at $95.

In [ ]:
qty = size_by_risk(
    account_value=10_000,
    risk_pct=0.02,       # risk 2% of account ($200)
    entry_price=100.0,
    stop_loss=95.0,      # risk per unit = $5
)
print(f"Position size: {qty:.1f} shares")
print(f"If stopped out: loss = {qty * 5:.0f} = {qty * 5 / 10_000 * 100:.1f}% of account")

In [ ]:
# Explore how position size changes with stop-loss distance
stop_distances = np.arange(1, 21, 0.5)
quantities = [
    size_by_risk(10_000, 0.02, 100.0, 100.0 - d)
    for d in stop_distances
]

fig, ax = plt.subplots()
ax.plot(stop_distances, quantities, linewidth=1.5)
ax.set_xlabel("Stop-Loss Distance ($)")
ax.set_ylabel("Position Size (shares)")
ax.set_title("Risk-Based Sizing: Tighter stops = larger positions")
ax.axhline(10, color="grey", linestyle="--", linewidth=0.8, label="Fixed qty=10")
ax.legend()
plt.tight_layout()
plt.show()

## 2. Fixed vs. Risk-Based Sizing in a Backtest

Let's compare the two approaches on the same strategy and data.

In [ ]:
# Generate oscillating price data
np.random.seed(42)
n = 150
base = 100 + 10 * np.sin(np.linspace(0, 6 * np.pi, n)) + np.cumsum(np.random.randn(n) * 0.3)
high = base + np.abs(np.random.randn(n) * 1.5)
low = base - np.abs(np.random.randn(n) * 1.5)
rows = []
for i in range(n):
    rows.append({
        "ts": datetime(2024, 1, 1) + timedelta(days=i),
        "open": base[i] + np.random.randn() * 0.3,
        "high": high[i],
        "low": low[i],
        "close": base[i],
        "volume": 1_000_000.0,
    })
candles = pd.DataFrame(rows)


class SwingTrader(BaseStrategy):
    """BUY when price < SMA, SELL when above."""

    def validate_parameters(self):
        pass

    def get_required_candle_count(self):
        return 20

    def compute(self, candles, as_of):
        closes = candles["close"].values
        ma = np.mean(closes[-20:])
        price = closes[-1]
        if price < ma * 0.97:
            return SignalOutput(direction=SignalDirection.BUY, confidence=0.8)
        elif price > ma * 1.03:
            return SignalOutput(direction=SignalDirection.SELL, confidence=0.8)
        return None

In [ ]:
# Fixed sizing: always trade 10 shares
p_fixed = Portfolio(
    initial_capital=10_000,
    quantity_per_trade=10,
    exit_rules=ExitRules(stop_loss_pct=0.05),
)
e_fixed = BacktestEngine(strategy=SwingTrader({}), portfolio=p_fixed, instrument_id="SYN")
r_fixed = e_fixed.run(candles)

# Risk-based sizing: risk 2% with 5% stop-loss
def risk_sizer(account_value, entry_price):
    stop = entry_price * 0.95  # 5% below entry
    return size_by_risk(account_value, risk_pct=0.02, entry_price=entry_price, stop_loss=stop)

p_risk = Portfolio(
    initial_capital=10_000,
    sizing_fn=risk_sizer,
    exit_rules=ExitRules(stop_loss_pct=0.05),
)
e_risk = BacktestEngine(strategy=SwingTrader({}), portfolio=p_risk, instrument_id="SYN")
r_risk = e_risk.run(candles)

print(f"{'Metric':<25} {'Fixed (qty=10)':>15} {'Risk-based (2%)':>15}")
print("-" * 55)
for key in ["total_trades", "total_return_pct", "win_rate_pct", "max_drawdown_pct", "sharpe_ratio"]:
    v1 = r_fixed.summary()[key]
    v2 = r_risk.summary()[key]
    print(f"{key:<25} {v1:>15} {v2:>15}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Equity curves
for result, label in [(r_fixed, "Fixed (qty=10)"), (r_risk, "Risk-based (2%)")]:
    cpnl = result.cumulative_pnl
    axes[0].plot(range(len(cpnl)), cpnl, linewidth=1.5, label=label)
axes[0].axhline(0, color="grey", linestyle="--", linewidth=0.5)
axes[0].set_title("Equity Curves")
axes[0].set_xlabel("Trade #")
axes[0].set_ylabel("Cumulative P&L ($)")
axes[0].legend()

# Position sizes comparison
fixed_sizes = [t.quantity for t in r_fixed.trades]
risk_sizes = [t.quantity for t in r_risk.trades]
x = np.arange(max(len(fixed_sizes), len(risk_sizes)))
width = 0.35
axes[1].bar(x[:len(fixed_sizes)] - width / 2, fixed_sizes, width, label="Fixed", alpha=0.7)
axes[1].bar(x[:len(risk_sizes)] + width / 2, risk_sizes, width, label="Risk-based", alpha=0.7)
axes[1].set_title("Position Sizes Per Trade")
axes[1].set_xlabel("Trade #")
axes[1].set_ylabel("Shares")
axes[1].legend()

plt.tight_layout()
plt.show()

## 3. How `sizing_fn` Works with Portfolio

The `sizing_fn` receives `(cash_balance, entry_price)` at each entry and returns
the quantity to trade. This means position sizes **adapt** to:

- Changing account equity (after wins/losses)
- Different price levels

```python
# Example: risk 1% with a 3% stop-loss
portfolio = Portfolio(
    initial_capital=50_000,
    sizing_fn=lambda cash, price: size_by_risk(
        account_value=cash,
        risk_pct=0.01,
        entry_price=price,
        stop_loss=price * 0.97,
    ),
    exit_rules=ExitRules(stop_loss_pct=0.03),
)
```

When `sizing_fn` is provided, `quantity_per_trade` is ignored.